# ABot-World：在 PAI-DSW 上的交互式世界模型

本 Notebook 是 [ABot-World](https://github.com/amap-cvlab/ABot-World) 在阿里云 PAI **交互式建模（DSW）** 中的使用说明，写法对齐 Notebook Gallery 案例（例如 [NVIDIA Cosmos](https://pai.console.aliyun.com/#/dsw-gallery/preview/deepLearning/cv/cosmos)）。

ABot-World 把单卡 GPU 变成可实时交互的世界模拟器：以一张参考图为起点，用键盘动作（WASD / IJKL）持续 rollout，默认 **704×1280、约 12–16 FPS**，峰值显存约 **19 GB**。

相关文档：

- [Notebook Gallery 使用说明](https://help.aliyun.com/zh/pai/dsw-gallery)
- [创建 DSW 实例](https://help.aliyun.com/zh/pai/create-and-manage-dsw-instances)
- [通过公网访问实例中的服务](https://help.aliyun.com/zh/pai/custom-services-access-configurations)
- [Model Gallery 使用案例汇总](https://help.aliyun.com/zh/pai/best-practices)


## 运行环境要求

**镜像：** 请使用 README 中已预装好运行环境的镜像。创建 DSW 时，镜像配置选择 **镜像地址**，填入下面地址（国内优先 ACR）。镜像内已安装 Ubuntu 22.04、CUDA 12.8、Python 3.12、PyTorch 2.8.0、FlashAttention、SageAttention、`lightx2v_kernel` 以及 `requirements.txt`，**不需再安装环境**。镜像只含运行环境，不含仓库代码和权重。

国内（阿里云 ACR）：

`crpi-56gxy7bfn4owmfmn.cn-zhangjiakou.personal.cr.aliyuncs.com/amap-cvlab/abot-world:v0-env`

海外（Docker Hub）：

`amapcvlab/abot-world:v0-env`

若为私有仓库，在镜像地址处填写仓库用户名和密码。不要在 DSW **里面** 再 `docker pull` / `docker run`——把上述地址当作实例镜像即可。

Notebook Kernel 请使用镜像内环境：`/opt/conda/envs/aworld/bin/python`。

**GPU 规格：** 推理使用 **单卡** 即可（当前发布不支持多卡并行）。显存峰值约 19 GB，系统内存建议 ≥ 64 GB。

| 场景 | 建议规格 |
| --- | --- |
| 推理 / Gradio Demo | 单卡 A100、A800、L20、H20 等，显存 80 GB 更从容；A10 24 GB 在显存上勉强可跑 |
| 精调 | 本 Notebook 不覆盖训练；当前开源为推理 |

建议申请 **>100 GB** 的外部存储（NAS、CPFS 或 OSS 挂载），用来存放预训练权重。仅使用免费系统盘时，实例停止超过 15 天数据会被清空。

工作目录默认为 `/mnt/workspace`。


## 0. 创建并打开 DSW 实例

若你已经在运行中的 DSW 里打开了本 Notebook，可跳过本节。

1. 登录 [PAI 控制台](https://pai.console.aliyun.com/)，选择地域与工作空间。
2. 左侧进入 **交互式建模（DSW）** → **新建实例**。
3. 按上一节选择 GPU 规格；**镜像配置** 选 **镜像地址**，填入 README 预装环境镜像（国内用 ACR 地址）。系统盘建议扩容，或挂载 NAS/OSS 到例如 `/mnt/data`。
4. 实例状态变为 **运行中** 后，单击 **打开**，进入 Notebook / WebIDE。
5. 将本文件放到工作目录（随仓库 clone，或从 Notebook Gallery「在 DSW 中打开」）。

从 Gallery 打开案例的步骤见 [Notebook Gallery 使用说明](https://help.aliyun.com/zh/pai/dsw-gallery)：选择工作空间与已创建的 DSW 实例，实例镜像、规格应与案例推荐环境一致。

**计费提示：** 公共资源实例在状态为运行中时即开始计费；关闭浏览器不会停止实例。用完请 **停止** 或 **删除**。


## 1. 环境配置

### Step1, 下载源码

考虑到仓库会持续更新，建议先跑通当前默认分支；需要复现某一版结果时再 `git checkout` 到指定 commit。

**下载方式 1：** 从 GitHub clone（实例需能访问 GitHub；网速慢时请在实例网络中配置 [专有网关](https://help.aliyun.com/zh/pai/dsw-network-configuration/)）。在 Terminal 或 Notebook 中等价于：

```
# 从 GitHub 下载 ABot-World 源码
%cd /mnt/workspace
!git clone https://github.com/amap-cvlab/ABot-World.git
%cd ABot-World
!pwd
```

**下载方式 2：** 若本 Notebook 已经位于 clone 好的 `ABot-World/dsw/` 目录中，下一格会自动检测到仓库根目录并跳过 clone。请**运行下一格**（不要只复制上面的 `%cd` 示例），它会设置后续所有步骤共用的 `REPO_ROOT`。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def _has_repo(p: Path) -> bool:
    return (p / "scripts" / "inference.py").is_file() and (p / "web_client" / "run.sh").is_file()


def locate_or_clone() -> Path:
    cwd = Path.cwd().resolve()
    for cand in (
        cwd,
        cwd.parent,
        Path("/mnt/workspace/ABot-World"),
        Path("/root/ABot-World"),
    ):
        if _has_repo(cand):
            print(f"已检测到 ABot-World 仓库，跳过 clone: {cand}")
            return cand

    workspace = Path("/mnt/workspace") if Path("/mnt/workspace").is_dir() else Path.home()
    dest = workspace / "ABot-World"
    if _has_repo(dest):
        print(f"已存在仓库: {dest}")
        return dest

    print(f"clone 到 {dest} ...")
    subprocess.check_call(
        ["git", "clone", "https://github.com/amap-cvlab/ABot-World.git", str(dest)]
    )
    return dest


REPO_ROOT = locate_or_clone()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.environ["PROJECT_ROOT"] = str(REPO_ROOT)
print("REPO_ROOT =", REPO_ROOT)
print("cwd       =", Path.cwd())


### Step2, 检查运行环境

确认当前 Kernel 能看到 GPU，且走的是镜像内预装环境（`/opt/conda/envs/aworld/bin/python`）。若 `torch` 导入失败，多半是 Kernel 没指到镜像里的 Python，而不是缺少依赖（见文末 FAQ）。


In [ ]:
import os
import shutil
import sys
from pathlib import Path

assert "REPO_ROOT" in globals(), "请先运行 Step1 的代码格"
os.chdir(REPO_ROOT)

print("python :", sys.version.replace("\n", " "))
print("which  :", shutil.which("python") or sys.executable)
print("cwd    :", Path.cwd())

!nvidia-smi

import torch
print("torch  :", torch.__version__)
print("cuda   :", torch.version.cuda)
print("gpu_ok :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu    :", torch.cuda.get_device_name(0))
    print("vram_gb:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


推荐镜像已按 README 装好运行依赖，**不要在 Notebook 里重新 pip install / 编译**。下一格只做导入检查；失败时先核对 Kernel 是否为 `/opt/conda/envs/aworld/bin/python`，而不是再装一遍环境。

`huggingface_hub` 与 `modelscope` 也已预装，可直接下载权重。


In [ ]:
import importlib
import sys

print("executable:", sys.executable)

required = ["torch", "flash_attn", "sageattention", "gradio", "modelscope", "huggingface_hub"]
optional = ["lightx2v_kernel"]
missing = []
for name in required:
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, "__version__", "ok")
        print(f"  {name}: {ver}")
    except Exception as exc:
        missing.append(name)
        print(f"  {name}: FAIL ({type(exc).__name__}: {exc})")

for name in optional:
    try:
        importlib.import_module(name)
        print(f"  {name}: ok")
    except Exception as exc:
        print(f"  {name}: 未导入（{type(exc).__name__}）。推理仍可用 --quant-type none")

if missing:
    raise ImportError(
        "预装环境检查失败: " + ", ".join(missing)
        + "。请把 Kernel 切到 /opt/conda/envs/aworld/bin/python 后 Restart。"
    )
print("环境已就绪，跳过安装。")


In [ ]:
# 推荐镜像已预装 SageAttention / lightx2v_kernel，无需编译。
print("跳过源码编译。")


### Step3, 下载预训练模型

权重 **不包含** 在预装环境镜像里，需要单独下载到 `checkpoints/ABot-World-0-5B-LF/`。国内环境优先 ModelScope。

需要的文件：

```text
checkpoints/ABot-World-0-5B-LF/
    ├── Wan2.2_VAE.pth
    ├── taew2_2.pth
    ├── models_t5_umt5-xxl-enc-bf16.pth
    ├── diffusion_pytorch_model.safetensors
    └── google/umt5-xxl/
```


In [ ]:
# 下载方式 1（推荐国内）: ModelScope
# 下载方式 2: HuggingFace，把 USE_MODELSCOPE 改为 False
import os
from pathlib import Path

os.chdir(REPO_ROOT)
USE_MODELSCOPE = True
CKPT = Path("checkpoints/ABot-World-0-5B-LF")
CKPT.mkdir(parents=True, exist_ok=True)

marker = CKPT / "diffusion_pytorch_model.safetensors"
if marker.is_file():
    print("权重已存在，跳过下载:", marker)
else:
    if USE_MODELSCOPE:
        from modelscope.hub.snapshot_download import snapshot_download

        snapshot_download(
            "amap_cvlab/ABot-World-0-5B-LF",
            local_dir=str(CKPT),
        )
    else:
        from huggingface_hub import snapshot_download

        snapshot_download(
            "acvlab/ABot-World-0-5B-LF",
            local_dir=str(CKPT),
        )
    print("下载完成:", CKPT.resolve())


In [ ]:
import os
from pathlib import Path

os.chdir(REPO_ROOT)
root = Path("checkpoints/ABot-World-0-5B-LF")
required = [
    "Wan2.2_VAE.pth",
    "taew2_2.pth",
    "models_t5_umt5-xxl-enc-bf16.pth",
    "diffusion_pytorch_model.safetensors",
    "google/umt5-xxl",
]
missing = [name for name in required if not (root / name).exists()]
print("checkpoint dir:", root.resolve())
if missing:
    raise FileNotFoundError("缺少文件: " + ", ".join(missing))
print("权重文件齐全。")
!du -sh checkpoints/ABot-World-0-5B-LF


## 2. 离线流式推理

`scripts/inference.py` 按 block 做因果生成：第 0 个 block 用参考图 encode 成 first latent，之后每个 block 3 个 latent，解码约 12 帧（首 block 约 9 帧）。默认参考图为 `web_client/datasets/images/example.png`，动作序列为 `web_client/datasets/actions/default_action.json`。

`--fps-blocks` 默认 3600（约 1 小时），下面先跑 **8 个 block** 做冒烟测试，大约数秒到十几秒视频，便于确认环境。

`--quant-type none` 走非量化路径，兼容 DSW 上常见的 A100/A800。镜像里的 `lightx2v_kernel` 主要按 Blackwell（如 RTX 5090）编译；若当前 GPU 上量化 kernel 可用，可改成 `fp8-per-token`。


In [ ]:
import os
from pathlib import Path

os.chdir(REPO_ROOT)
Path("outputs").mkdir(exist_ok=True)

# 8 blocks ≈ 数秒视频；确认跑通后再加大
FPS_BLOCKS = 8
QUANT_TYPE = "none"

!python scripts/inference.py --quant-type {QUANT_TYPE} --fps-blocks {FPS_BLOCKS}


In [ ]:
import os
from pathlib import Path
from IPython.display import Video, display

os.chdir(REPO_ROOT)
videos = sorted(
    Path("outputs/profile_bench").glob("**/*.mp4"),
    key=lambda p: p.stat().st_mtime,
)
if not videos:
    raise FileNotFoundError("未找到 outputs/profile_bench/**/*.mp4，请先跑通上一格。")
video_path = videos[-1]
print("video:", video_path)
display(Video(str(video_path), embed=True, width=640))


## 3. Gradio 交互式 Demo

本地入口是 `bash web_client/run.sh`，服务监听 `0.0.0.0:2233`。模型在后台加载，页面出现 **「唤醒你的世界」** 可点后再开始交互。

**与上一节不要同时跑：** 离线推理和 Gradio 都会占用同一张 GPU。若刚跑完推理，先重启 Kernel，或确认旧的 Python 推理进程已退出。

### 在 DSW 里访问 Web UI

DSW 实例默认没有公网 IP。任选其一：

1. **自定义服务（推荐）**：实例 **访问配置** → 添加自定义服务，监听端口 `2233`。需要公网访问时勾选公网访问，并配置 NAT 网关、EIP 与安全组入方向 TCP `2233`。详见 [通过公网访问实例中的服务](https://help.aliyun.com/zh/pai/custom-services-access-configurations)。
2. **VPC 内访问**：同 VPC 的 ECS 浏览器打开 `http://<DSW私网IP>:2233`。
3. Jupyter 若提供 proxy，可尝试 `.../proxy/2233/`（视当前 DSW 版本而定）。

浏览器建议使用 **Chrome**。


In [ ]:
import os
import subprocess
import time
from pathlib import Path

os.chdir(REPO_ROOT)
Path("outputs").mkdir(exist_ok=True)
log_path = Path("outputs/gradio_dsw.log")

# 若已有实例占用 2233，先不要重复启动
env = os.environ.copy()
env["CUDA_ID"] = env.get("CUDA_ID", "0")
env["ABOTWORLD_DEBUG_FRONTEND"] = "0"

log_f = open(log_path, "ab")
proc = subprocess.Popen(
    ["bash", "web_client/run.sh"],
    cwd=str(REPO_ROOT),
    stdout=log_f,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True,
)
print("Gradio pid =", proc.pid)
print("log        =", log_path.resolve())
print("等待服务监听 2233 ...")
for i in range(30):
    time.sleep(2)
    if proc.poll() is not None:
        print("进程已退出，请查看日志:")
        print(log_path.read_text(encoding="utf-8", errors="replace")[-4000:])
        break
    try:
        import socket

        with socket.create_connection(("127.0.0.1", 2233), timeout=1):
            print("已监听 http://0.0.0.0:2233")
            break
    except OSError:
        print(f"  ... {i*2+2}s")
else:
    print("仍未监听到端口，模型可能仍在加载，继续看日志即可。")


### 界面操作

1. 等待状态从「模型加载中」变为可点击 **唤醒你的世界**。
2. 在画廊中选一张参考图（或使用默认 `example.png`），确认 / 编辑 Prompt。
3. 点击 **唤醒你的世界** 开始流式生成。
4. 键盘：
   - **WASD**：水平移动
   - **IJKL**：另一组视角/运动控制
   - 对向键互斥（W 与 S、A 与 D、I 与 K、J 与 L）
5. 停止生成后，可在页面中查看已保存的视频；文件也在仓库 `outputs/` 下。

场景与默认 Prompt 配置见 `web_client/scene_presets.yaml`。


In [ ]:
# 查看 Gradio 最近日志（模型加载较慢时反复运行本格即可）
from pathlib import Path

log_path = Path("outputs/gradio_dsw.log")
if log_path.is_file():
    text = log_path.read_text(encoding="utf-8", errors="replace")
    print(text[-3000:] if len(text) > 3000 else text)
else:
    print("还没有 outputs/gradio_dsw.log")


## 4. 常见问题

### Q：运行时提示找不到 `torch` 或其他 Python 模块？

推荐镜像已经预装依赖。这通常是 Notebook Kernel 没有用镜像里的 Python。处理方式：

1. 确认实例镜像是 README 中的 `abot-world:v0-env`（ACR 或 Docker Hub）。
2. Kernel 选 `/opt/conda/envs/aworld/bin/python`，然后 **Restart Kernel**，从 Step1 再跑。
3. 不要在 Notebook 里重新 `pip install` 整套环境。

### Q：可以在 DSW 里 `docker pull` / `docker run` 吗？

不需要。创建实例时把 README 镜像填进 **镜像地址** 即可。DSW 本身已在容器中运行，一般不要再嵌套 Docker。

### Q：A10 24 GB 能不能跑？

峰值显存约 19 GB，A10 在容量上接近上限。优先选 80 GB 规格；若必须用 24 GB，保持 `--quant-type none`、不要同时开 Gradio 和离线推理，并关闭其它占 GPU 的进程。

### Q：`lightx2v_kernel` 报 CUDA architecture / kernel 错误？

镜像中的量化 kernel 面向 Blackwell。在 DSW 的 A100/A800 等卡上先用 `--quant-type none`。

### Q：GitHub / HuggingFace 下载很慢或失败？

权重改走 ModelScope；代码 clone 为实例配置 [专有网关 + NAT/EIP](https://help.aliyun.com/zh/pai/dsw-network-configuration/)。不要把大权重放到会 15 天清空的免费系统盘，请挂载 NAS/OSS。

### Q：Gradio 页面打不开？

确认本 Notebook 已成功监听 `2233`，并在 DSW **访问配置** 里添加了对应自定义服务；公网访问还需安全组放行。加载模型期间页面能开但按钮不可用，属于预期。


## 参考链接

- 项目主页与安装：<https://github.com/amap-cvlab/ABot-World>
- 技术报告：<https://arxiv.org/abs/2607.19191>
- ModelScope 权重：<https://modelscope.cn/models/amap_cvlab/ABot-World-0-5B-LF>
- HuggingFace 权重：<https://huggingface.co/acvlab/ABot-World-0-5B-LF>
- [Notebook Gallery](https://help.aliyun.com/zh/pai/dsw-gallery) · [创建 DSW 实例](https://help.aliyun.com/zh/pai/create-and-manage-dsw-instances) · [自定义服务端口](https://help.aliyun.com/zh/pai/custom-services-access-configurations)
